In [2]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 25.2 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 30.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np

In [2]:
prices = pd.read_csv("sell_prices.csv")
calendar = pd.read_csv("calendar.csv")
sales = pd.read_csv("sales_train_evaluation.csv")

In [3]:
#Make assumption that food items will fluctuat
foods_items = sales[sales["cat_id"] == "FOODS"]["item_id"].unique()
foods_prices = prices[prices["item_id"].isin(foods_items)]

In [5]:
# 3. Compute price volatility per item (across all stores)
#    Higher coefficient of variation (CV) = more price movement = promo candidate
# ---------------------------------------------------------
price_stats = (
    foods_prices.groupby("item_id")["sell_price"]
    .agg(mean_price="mean", std_price="std", n_price_points="count")
    .reset_index()
)
price_stats["cv"] = price_stats["std_price"] / price_stats["mean_price"]

# Require enough price history to be meaningful
price_stats = price_stats[price_stats["n_price_points"] > 50]

# Top 20 most price-volatile FOODS items
top_volatile = price_stats.sort_values("cv", ascending=False).head(20)
print("Top 20 price-volatile FOODS items:")
print(top_volatile[["item_id", "mean_price", "cv", "n_price_points"]])


Top 20 price-volatile FOODS items:
          item_id  mean_price        cv  n_price_points
908   FOODS_3_296    1.777010  0.267782             592
917   FOODS_3_305    3.537660  0.202580            2820
93    FOODS_1_095    3.822748  0.193026            2820
82    FOODS_1_084    6.168478  0.186766            2628
416   FOODS_2_202    2.888314  0.184701            2337
77    FOODS_1_079    6.285500  0.183287            2318
104   FOODS_1_107    6.166099  0.182711            2776
74    FOODS_1_076    6.438653  0.181694            2814
32    FOODS_1_034    6.441491  0.181523            2690
51    FOODS_1_053    6.457340  0.181304            2793
123   FOODS_1_126    2.369309  0.179541            2242
96    FOODS_1_098    5.408037  0.173603            1890
827   FOODS_3_215    1.044755  0.165305            2736
617   FOODS_3_004    5.089413  0.162510            1601
167   FOODS_1_171    5.581824  0.154929            1897
1112  FOODS_3_500    2.405135  0.153826             742
1211  FOODS_3

In [8]:
# 4. For a shortlist of candidates, check if sales actually spike
#    when price drops (not just that price is volatile)
# ---------------------------------------------------------
def demand_price_correlation(item_id, store_id="CA_1"):
    """
    Returns the correlation between weekly price and weekly sales
    for one item at one store. Strong negative correlation = 
    demand rises when price falls = good promo candidate.
    """
    # Sales row for this item/store
    sales_row = sales[
        (sales["item_id"] == item_id) & (sales["store_id"] == store_id)
    ]
    if sales_row.empty:
        return None

    day_cols = [c for c in sales.columns if c.startswith("d_")]
    daily_sales = sales_row[day_cols].T
    daily_sales.columns = ["units_sold"]
    daily_sales["d"] = daily_sales.index

    # Map d_ columns to actual weeks via calendar
    daily_sales = daily_sales.merge(calendar[["d", "wm_yr_wk"]], on="d")
    weekly_sales = daily_sales.groupby("wm_yr_wk")["units_sold"].sum().reset_index()

    item_prices = prices[
        (prices["item_id"] == item_id) & (prices["store_id"] == store_id)
    ][["wm_yr_wk", "sell_price"]]

    merged = weekly_sales.merge(item_prices, on="wm_yr_wk")
    if len(merged) < 10:
        return None

    corr = merged["units_sold"].corr(merged["sell_price"])
    return corr

print("\nChecking demand-price correlation for top candidates (store CA_1):")
results = []
for item_id in top_volatile["item_id"].head(10):
    corr = demand_price_correlation(item_id)
    if corr is not None:
        results.append((item_id, corr))

results_df = pd.DataFrame(results, columns=["item_id", "price_sales_corr"])
results_df = results_df.sort_values("price_sales_corr")  # most negative first
print(results_df)

print("""
How to read this:
- price_sales_corr close to -1: strong promo pattern (sales rise sharply as price drops)
- price_sales_corr near 0: little relationship (steady seller, not promo-driven)
- Pick 1-2 items from the top of results_df (most negative correlation) as your
  forecasting subjects.
""")



Checking demand-price correlation for top candidates (store CA_1):


/usr/local/Cellar/jupyterlab/4.6.2/libexec/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/Cellar/jupyterlab/4.6.2/libexec/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


       item_id  price_sales_corr
1  FOODS_3_305         -0.526304
4  FOODS_2_202         -0.442384
7  FOODS_1_076         -0.315410
2  FOODS_1_095         -0.190083
8  FOODS_1_034         -0.171539
3  FOODS_1_084          0.003256
5  FOODS_1_079          0.016002
6  FOODS_1_107          0.016586
9  FOODS_1_053          0.163842
0  FOODS_3_296               NaN

How to read this:
- price_sales_corr close to -1: strong promo pattern (sales rise sharply as price drops)
- price_sales_corr near 0: little relationship (steady seller, not promo-driven)
- Pick 1-2 items from the top of results_df (most negative correlation) as your
  forecasting subjects.



In [9]:
# ---------------------------------------------------------
# 5. Check whether the promo pattern for our chosen item holds
#    across ALL stores, not just CA_1
# ---------------------------------------------------------
CHOSEN_ITEM = "FOODS_3_305"
all_stores = sales["store_id"].unique()
 
store_results = []
for store_id in all_stores:
    corr = demand_price_correlation(CHOSEN_ITEM, store_id=store_id)
    if corr is not None:
        store_results.append((store_id, corr))
 
store_results_df = pd.DataFrame(store_results, columns=["store_id", "price_sales_corr"])
store_results_df = store_results_df.sort_values("price_sales_corr")
 
print(f"\nPrice-sales correlation for {CHOSEN_ITEM} across all stores:")
print(store_results_df)
print(f"\nMean correlation across stores: {store_results_df['price_sales_corr'].mean():.3f}")
print(f"Std dev across stores: {store_results_df['price_sales_corr'].std():.3f}")
 
print("""
How to read this:
- If most/all stores show a similar negative correlation: the promo pattern is
  real and consistent, aggregating nationwide is reasonable.
- If correlations vary widely (some strongly negative, some near zero or positive):
  the pattern is store/region-specific, worth noting in your writeup, and you may
  want to pick a single representative store (or a cluster of similar stores)
  rather than aggregating everything together.
""")
 



Price-sales correlation for FOODS_3_305 across all stores:
  store_id  price_sales_corr
5     TX_2         -0.783573
1     CA_2         -0.756266
4     TX_1         -0.740402
6     TX_3         -0.714671
2     CA_3         -0.651265
9     WI_3         -0.633690
7     WI_1         -0.552521
8     WI_2         -0.551902
0     CA_1         -0.526304
3     CA_4         -0.516851

Mean correlation across stores: -0.643
Std dev across stores: 0.102

How to read this:
- If most/all stores show a similar negative correlation: the promo pattern is
  real and consistent, aggregating nationwide is reasonable.
- If correlations vary widely (some strongly negative, some near zero or positive):
  the pattern is store/region-specific, worth noting in your writeup, and you may
  want to pick a single representative store (or a cluster of similar stores)
  rather than aggregating everything together.

